# Chilbolton Climatology — Student 3: Surface Pressure

This is **Student 3's notebook** as part of a group project on the climatology of Chilbolton Observatory.
The four students are each analysing a different meteorological variable:
- **Student 1**: Rainfall
- **Student 2**: Air temperature and relative humidity
- **Student 3 (you)**: Surface pressure
- **Student 4**: Wind speed and direction

## Learning Objectives
By the end of this notebook you should be able to:
- Load and quality-control pressure data from NetCDF files
- Characterise the statistical distribution of atmospheric pressure
- Identify high and low pressure events from the observational record
- Describe the seasonal cycle of pressure at Chilbolton

## Background
Surface air pressure at a mid-latitude site like Chilbolton (51.1°N, 1.4°W) is primarily driven by
the passage of synoptic weather systems — high-pressure anticyclones typically bring dry settled weather,
while low-pressure depressions bring cloud, rain and wind.

- `air_pressure` is stored in **hPa** (equivalent to mbar); typical values are 960–1040 hPa
- QC flag: `qc_flag_air_pressure` (value 1 = good, anything else = suspect or bad)

## Setup — Run This First

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import netCDF4 as nc4
import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 200)
print('Libraries loaded.')

In [ ]:
def load_pressure(root: str, qc_good_only: bool = True) -> pd.DataFrame:
    """
    Load all pressure NetCDF files from root and return a time-indexed DataFrame.

    Columns returned:
        time     : UTC timestamp (pandas Timestamp)
        pressure : surface air pressure in hPa
    """
    files = sorted(Path(root).rglob('*.nc'))
    if not files:
        raise FileNotFoundError(f'No .nc files found under {root}')

    chunks = []
    for f in files:
        with nc4.Dataset(str(f)) as nc:
            unix     = nc.variables['time'][:].data.copy().astype(np.float64)
            pressure = nc.variables['air_pressure'][:].data.copy().astype(np.float64)
            qc       = nc.variables['qc_flag_air_pressure'][:].data.copy().astype(np.int8)

        if qc_good_only:
            pressure[(qc != 0) & (qc != 1)] = np.nan

        chunks.append(pd.DataFrame({'unix': unix, 'pressure': pressure}))

    df = pd.concat(chunks, ignore_index=True).sort_values('unix').reset_index(drop=True)
    df['time'] = pd.to_datetime(df['unix'], unit='s', utc=True)
    return df.drop(columns='unix')


print('Helper functions defined.')

## Task 1: Load and Explore the Data

**What to do:**
1. Set `ROOT_PATH` to the pressure data directory and run the cell
2. Print basic statistics — min, mean, max, standard deviation
3. How does this compare to the standard atmosphere (1013.25 hPa)?

**Questions:**
- What is the lowest pressure recorded in the dataset? When did it occur?
- What is the highest pressure recorded? When did it occur?

In [ ]:
# ===== EDIT THIS =====
ROOT_PATH = '/data/wexp/cwalden/pressure'
QC_GOOD_ONLY = True
PLOT_THEME = 'light'  # 'light' or 'dark'
# =====================

plt.style.use('dark_background' if PLOT_THEME == 'dark' else 'default')

df = load_pressure(ROOT_PATH, qc_good_only=QC_GOOD_ONLY)

print(f'Loaded {len(df):,} records')
print(f'Date range: {df["time"].min().date()} to {df["time"].max().date()}')
print(f'Pressure (hPa): min={df["pressure"].min():.1f}  mean={df["pressure"].mean():.1f}  '
      f'max={df["pressure"].max():.1f}  std={df["pressure"].std():.2f}')

# Find the lowest and highest pressure events
idx_min = df['pressure'].idxmin()
idx_max = df['pressure'].idxmax()
print(f'\nLowest:  {df.loc[idx_min, "pressure"]:.1f} hPa at {df.loc[idx_min, "time"]}')
print(f'Highest: {df.loc[idx_max, "pressure"]:.1f} hPa at {df.loc[idx_max, "time"]}')

## Task 2: Pressure Distribution

**What to do:**
Plot the distribution of pressure values. Pressure at a mid-latitude site is approximately
normally distributed.

1. Plot a histogram of all pressure values (use ~100 bins)
2. Overlay a normal distribution curve fitted to the data
3. Mark the 5th and 95th percentiles on the plot — these loosely correspond to deep low/strong high events

**Hints:**
- Use `scipy.stats.norm.fit()` to fit a normal distribution: `mu, sigma = scipy.stats.norm.fit(data)`
- Then plot `scipy.stats.norm.pdf(x, mu, sigma)` over the histogram
- Use `ax.axvline()` to mark the percentiles

In [ ]:
import scipy.stats

p_clean = df['pressure'].dropna()

# TODO: plot histogram of pressure values
# TODO: fit and overlay a normal distribution
# TODO: mark the 5th and 95th percentiles
p5  = p_clean.quantile(0.05)
p95 = p_clean.quantile(0.95)
print(f'5th percentile:  {p5:.1f} hPa')
print(f'95th percentile: {p95:.1f} hPa')

## Task 3: Monthly Climatology

**What to do:**
Investigate whether pressure has a seasonal cycle.

1. Compute the climatological monthly mean and standard deviation of pressure
2. Plot mean ± 1 std as a function of month
3. Is there a seasonal cycle? Compare your result with published climatological data for southern England

**Hint:** Group by `time.dt.month` to compute per-month statistics.

In [ ]:
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# TODO: compute monthly climatology (mean and std) of pressure
# TODO: plot monthly mean ± 1 std

# Example groupby structure:
# monthly = df.groupby(df['time'].dt.month)['pressure'].agg(['mean', 'std', 'count'])
# monthly.index = MONTH_NAMES
# print(monthly)

## Task 4: Identifying Weather Events

**What to do:**
Use the pressure time series to identify synoptic weather events.

1. Resample the data to **hourly** means (to smooth noise)
2. Plot a 3-month section of the time series (your choice of months)
3. Identify and annotate:
   - At least 2 **low pressure** events (pressure < 5th percentile = `p5`)
   - At least 2 **high pressure** events (pressure > 95th percentile = `p95`)
4. Can you match any of these events to actual weather events using news/weather archives?

**Hints:**
- Resample: `df_hourly = df.set_index('time').resample('h')['pressure'].mean().reset_index()`
- To select a date range: `mask = (df_hourly['time'] >= '2023-01-01') & (df_hourly['time'] < '2023-04-01')`
- Use `ax.axhline(p5, linestyle='--')` to mark thresholds

In [ ]:
# Resample to hourly means
df_hourly = df.set_index('time').resample('h')['pressure'].mean().reset_index()

# TODO: choose a 3-month window and plot the time series
# TODO: mark low and high pressure thresholds
# TODO: annotate notable events

## Task 5 (Optional): Annual Means and Long-term Trend

**What to do:**
If the dataset spans multiple years, compute the annual mean pressure for each year and
check whether there is any long-term trend.

1. Compute annual mean pressure
2. Fit a linear trend using `numpy.polyfit()`
3. Is the trend statistically significant? (You can use `scipy.stats.linregress()` to get the p-value)

**Note:** A real climatic trend in pressure would require many decades of data. This exercise
is more about learning the method than expecting to find a robust signal.

In [ ]:
# TODO: compute annual means and fit a linear trend
# annual = df.groupby(df['time'].dt.year)['pressure'].mean()
# slope, intercept, r, p, se = scipy.stats.linregress(annual.index, annual.values)
# print(f'Trend: {slope:.3f} hPa/year  p-value: {p:.3f}')

## Reflection Questions

1. **Distribution shape**: Is the pressure distribution symmetric? Does a normal distribution fit well? Are there any tails that suggest extreme events?
2. **Seasonal cycle**: Did you find a seasonal cycle? Is it large compared to the day-to-day variability (standard deviation)?
3. **Weather events**: Can you identify specific named storms or blocking anticyclones in the time series?
4. **Comparison with your teammates**: Does the pressure time series match the rainfall or wind patterns found by Student 1 and Student 4? (Low pressure = more rain and wind?)
5. **Physical reasoning**: Why might surface pressure vary with season at Chilbolton?